In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

In [3]:
def runge(x):
    return (1.0/(1.0+25*x**2))
    

def design_matrix(x, degree):
    X = np.vander(np.asarray(x).ravel(), degree +1 , increasing=True)
    return X

rng = np.random.default_rng(2026)
n = 100
sigma = 0.1
x = np.sort(rng.uniform(-1,1,n))
y = runge(x) + rng.normal(0, sigma,n)


def exercisedata(x, y, d):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2026)
    scaler = StandardScaler()
    Xtr = design_matrix(x_train, d)
    
    Xtr_sc = np.copy(Xtr)
    Xtr_sc[:,1:] = scaler.fit_transform(Xtr[:,1:]) 
    

    Xte = design_matrix(x_test, d)
    Xte_sc = np.copy(Xte)
    Xte_sc[:,1:] = scaler.transform(Xte[:,1:]) 
    theta = np.linalg.lstsq(Xtr_sc, y_train, rcond=None)[0]

    return theta, Xtr_sc, Xte_sc, y_train, y_test


def optimiser_step(method, theta, g, state, t, gamma, beta=0.9, rho=0.99,beta1=0.9, beta2=0.999, eps=1e-8):
    """One update of theta from the gradient g at step t = 1, 2, ...; state carries the running quantities."""
    if method == "plain":                                    # Eq. (4.10)
        return theta - gamma * g, state
    if method == "momentum":                                 # Eq. (4.28)
        state["v"] = v = beta * state.get("v", 0.0) + gamma * g 
        return theta - v, state
    if method == "adagrad":                                  # Eqs. (4.42) and (4.45)
        state["r"] = r = state.get("r", 0.0) + g * g
        return theta - gamma * g / (np.sqrt(r) + eps), state
    if method == "rmsprop":                                  # Eqs. (4.47) and (4.48)
        state["r"] = r = rho * state.get("r", 0.0) + (1.0 - rho) * g * g 
        return theta - gamma * g / (np.sqrt(r) + eps), state
    if method == "adam":                                     # Eqs. (4.51), (4.52), (4.54), (4.55)
        state["m"] = m =  beta1 * state.get("m", 0.0) + (1.0 - beta1) * g
        state["r"] = r = beta2 * state.get("r", 0.0) + (1.0 - beta2) * g * g
        m_hat, r_hat = m / (1.0 - beta1**t), r / (1.0 - beta2**t)
        return theta - gamma * m_hat / (np.sqrt(r_hat) + eps), state
    raise ValueError(f"unknown method {method}")


def optimise(grad, theta0, method, gamma, num_iters=1000, tol=1e-8, **kw):
    """Full-gradient loop; returns all iterates and the number of steps taken."""
    theta, state = np.array(theta0, dtype=float), {}
    
    history = [theta.copy()]
    for t in range(1, num_iters + 1):
        with np.errstate(over="ignore", invalid="ignore"):
            g = grad(theta)
        #g = grad(theta)
        if not np.all(np.isfinite(g)):
            break
        theta, state = optimiser_step(method, theta, g, state, t, gamma, **kw)
        if not np.all(np.isfinite(theta)):
            break

        history.append(theta.copy())
        if np.linalg.norm(g) < tol:
            break
    return np.array(history), t


def lasso_gradient(theta, X, y, lam):
    n = len(y)

    mse_grad = (2 / n) * X.T @ (X @ theta - y)

    penalty_grad = lam * np.sign(theta)
    penalty_grad[0] = 0.0  # Do not penalize intercept

    return mse_grad + penalty_grad

In [4]:
lambdas = np.logspace(-6, 0, 100)
degrees = np.arange(1, 16)

gamma = 0.01
num_iters = 10000

best_results = []

for d in degrees:

    _, Xtr_sc, Xte_sc, yr_train, yr_test = exercisedata(x, y, d)

    for lam in lambdas:

        theta0 = np.zeros(Xtr_sc.shape[1])

        # Lasso gradient for current lambda
        grad = lambda theta: lasso_gradient(
            theta, Xtr_sc, yr_train, lam
        )

        # Optimize using Adam
        history, steps = optimise(
            grad,
            theta0,
            method="adam",
            gamma=gamma,
            num_iters=num_iters
        )

        theta = history[-1]

        # Calculate test MSE
        yr_test_pred = Xte_sc @ theta
        test_mse = mean_squared_error(yr_test, yr_test_pred)

        best_results.append((d, lam, test_mse))

# Find best combination
best_degree, best_lambda, best_mse = min(
    best_results, key=lambda result: result[2]
)

print("Beste grad:", best_degree)
print("Beste lambda:", best_lambda)
print("Laveste test-MSE:", best_mse)

Beste grad: 8
Beste lambda: 0.00020092330025650479
Laveste test-MSE: 0.017449299908906608
